# 02. Write와 Select

Write는 정보를 외부 기록에 남기는 단계이고, Select는 현재 질문에 필요한 기록만 고르는 단계입니다.

## 실행 환경 준비

실습 파일은 `labs/part2/ch08-context-engineering/notebooks/02-Write와-Select.ipynb`입니다. 새 커널에서 시작해 첫 코드 셀로 패키지 버전을 맞춘 뒤, 설명과 코드를 위에서부터 순서대로 실행합니다. `.env`는 현재 작업 폴더의 상위 경로에서도 찾습니다.

In [1]:
# %pip은 현재 노트북 커널에 패키지를 설치하는 Jupyter 전용 명령입니다.
# -q는 설치 로그를 줄이고, == 뒤 숫자는 수업에서 검증한 버전을 고정합니다.
%pip install -q "langchain==1.4.0" "langchain-openai==1.6.0" "python-dotenv==1.2.3" "tiktoken==0.14.0"

Note: you may need to restart the kernel to use updated packages.


## 1. API 연결 준비

각 노트북은 빈 커널에서 시작할 수 있도록 연결 코드를 포함합니다.

In [2]:
# find_dotenv는 현재 폴더부터 상위 폴더를 검색해 .env 경로를 찾습니다.
from dotenv import find_dotenv, load_dotenv
import os

# usecwd=True는 노트북을 실행한 폴더를 검색 시작점으로 지정합니다.
# 키 값은 출력하지 않고 존재 여부만 확인합니다.
env_path = find_dotenv(usecwd=True)
if not env_path:
    raise FileNotFoundError("현재 폴더의 상위 경로에서 .env를 찾지 못했습니다.")
load_dotenv(env_path)
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(".env에 OPENAI_API_KEY가 없습니다.")
print(".env와 API 키 확인 완료")

.env와 API 키 확인 완료


### 모델 객체를 만듭니다

`.env` 확인이 끝난 뒤 모델 이름과 출력 제한을 지정합니다. 이 셀은 아직 API에 질문을 보내지 않습니다.

In [3]:
# init_chat_model은 provider:model 형식으로 채팅 모델 객체를 만듭니다.
# model은 이후 셀에서 invoke()를 호출할 때 사용합니다.
from langchain.chat_models import init_chat_model

# openai:는 공급자 이름이며, reasoning_effort="none"은 별도 추론 토큰을 사용하지 않는 설정입니다.
# max_tokens=300은 생성할 출력 토큰 수의 상한입니다.
model = init_chat_model(
    "openai:gpt-5.4-mini",
    reasoning_effort="none",
    max_tokens=300,
)
print("LangChain 모델 준비 완료")

LangChain 모델 준비 완료


## 2. 기록을 JSON으로 저장하고 다시 읽습니다

예제 데이터는 코드 안에 직접 적습니다. 저장 파일은 운영체제의 임시 폴더에 만들기 때문에 노트북을 어느 폴더에서 열어도 실행할 수 있습니다.

In [4]:
# records의 각 사전은 문서 식별자, 주제, 본문을 나타냅니다.
import json
from pathlib import Path
from tempfile import gettempdir

records = [
    {"id": "E01-p2", "topic": "휴관", "text": "매주 월요일 휴관"},
    {"id": "E01-p3", "topic": "대출", "text": "7권, 14일 대출"},
    {"id": "E01-p5", "topic": "주차", "text": "90분 무료"},
]
print("저장할 기록:", len(records))

저장할 기록: 3


### 임시 파일에 기록을 씁니다

`gettempdir()`를 사용하므로 노트북을 어느 폴더에서 열어도 같은 방식으로 실행됩니다. 출력된 경로에서 실제 저장 위치를 확인합니다.

In [5]:
# ensure_ascii=False는 한국어를 그대로 JSON에 기록합니다.
memory_path = Path(gettempdir()) / "ch08-memory.json"
json_text = json.dumps(records, ensure_ascii=False)
memory_path.write_text(json_text, encoding="utf-8")
print("임시 폴더에 저장한 파일:", memory_path.name)

임시 폴더에 저장한 파일: ch08-memory.json


### 파일을 다시 읽어 중간 결과를 확인합니다

`read_text()`는 문자열을 반환하고, `json.loads()`는 그 문자열을 파이썬 목록으로 복원합니다.

In [6]:
# loaded에는 파일에서 복원한 기록 목록이 저장됩니다.
saved_text = memory_path.read_text(encoding="utf-8")
loaded = json.loads(saved_text)
print("다시 읽은 기록:", len(loaded))

다시 읽은 기록: 3


## 3. 질문에 필요한 기록만 선택합니다

여기서는 학생이 식별자를 직접 지정합니다. 자동 검색은 Ch 9에서 다룹니다.

In [7]:
# selected_ids는 이번 질문에 필요한 문서 ID 모음입니다.
# 각 기록을 순서대로 검사해 일치하는 기록만 selected에 추가합니다.
selected_ids = {"E01-p3", "E01-p5"}
selected = []
for row in loaded:
    if row["id"] in selected_ids:
        selected.append(row)

for row in selected:
    print(row["id"], row["topic"], row["text"])

E01-p3 대출 7권, 14일 대출
E01-p5 주차 90분 무료


## 4. 선택한 기록으로 답을 생성합니다

모델에는 저장소 전체 대신 선택한 두 기록만 전달합니다.

In [8]:
# context_lines에 출처 ID와 내용을 한 줄씩 추가합니다.
context_lines = []
for row in selected:
    line = f"[{row['id']}] {row['text']}"
    context_lines.append(line)

context = "\n".join(context_lines)
prompt = f"""근거:
{context}

질문: 대출 권수와 주차 무료 시간을 알려 주세요."""
print(prompt)

근거:
[E01-p3] 7권, 14일 대출
[E01-p5] 90분 무료

질문: 대출 권수와 주차 무료 시간을 알려 주세요.


### 모델에 전달할 메시지를 만듭니다

`system`에는 답변 규칙을, `user`에는 앞에서 만든 질문을 넣습니다. 셀을 실행한 뒤 역할 순서를 확인합니다.

In [9]:
# messages는 역할과 내용을 짝지은 목록입니다.
# system 지시를 user 질문보다 먼저 둡니다.
messages = [
    ("system", "제공된 근거만 사용하고 문장마다 식별자를 붙이세요."),
    ("user", prompt),
]
roles = []
for role, content in messages:
    roles.append(role)
print("메시지 역할:", roles)

메시지 역할: ['system', 'user']


### 모델을 한 번 호출합니다

`invoke()`의 반환값은 `AIMessage`입니다. 사용량 정보가 없으면 이후 비교를 할 수 없으므로 즉시 오류를 냅니다.

In [10]:
# response는 모델의 답과 메타데이터를 가진 AIMessage입니다.
response = model.invoke(messages)
usage = response.usage_metadata

if usage is None:
    raise RuntimeError("모델 사용량 메타데이터가 없습니다.")

### 답과 실제 토큰 수를 관찰합니다

먼저 답변 원문을 읽고, 다음 두 줄에서 API가 보고한 입력·출력 토큰 수를 확인합니다. 답변 문구와 수치는 실행마다 달라질 수 있습니다.

In [11]:
# text는 답변 문자열이고 usage는 실제 호출의 토큰 사용량입니다.
print(response.text)
print("input_tokens:", usage.get("input_tokens"))
print("output_tokens:", usage.get("output_tokens"))

대출은 **7권**입니다. [E01-p3]  
주차 무료 시간은 **90분**입니다. [E01-p5]
input_tokens: 71
output_tokens: 36


## 5. 선택 ID 한 개를 줄여 비교합니다

기본 선택에는 대출과 주차가 들어 있습니다. 주차 ID 하나를 제외한 뒤 선택된 기록 수와 주제를 비교합니다.

In [12]:
# changed_ids에서는 E01-p5 하나만 제외합니다.
changed_ids = {"E01-p3"}
changed_selected = []
for row in loaded:
    if row["id"] in changed_ids:
        changed_selected.append(row)

print("기본 선택 수:", len(selected))
print("변경 선택 수:", len(changed_selected))
print("변경 후 주제:", changed_selected[0]["topic"])

기본 선택 수: 2
변경 선택 수: 1
변경 후 주제: 대출


## 정리

외부 기록은 저장만으로 모델에 전달되지 않습니다. 질문에 맞는 항목을 선택하여 호출 입력에 넣어야 합니다.

## 실패 재현: 저장한 기록을 선택하지 않습니다

파일에 기록이 있어도 선택 결과가 비어 있으면 모델에 전달할 근거가 없습니다.

In [13]:
# rows는 선택된 근거 목록이며, 빈 목록이면 모델에 줄 정보가 없습니다.
# -> None은 이 함수가 값을 반환하지 않고 검사만 한다는 뜻입니다.
def require_context(rows: list[dict[str, str]]) -> None:
    if not rows:
        raise ValueError("선택한 근거가 없습니다.")

### 빈 목록으로 실패를 관찰합니다

정상 선택값 대신 빈 목록 하나만 전달합니다. 오류 메시지가 문제 원인을 설명하는지 확인합니다.

In [14]:
# 빈 목록은 선택된 근거가 하나도 없다는 뜻입니다.
try:
    require_context([])
except ValueError as error:
    print("관찰한 실패:", error)
else:
    raise AssertionError("빈 근거 오류가 발생해야 합니다.")

관찰한 실패: 선택한 근거가 없습니다.


## 수정: 선택한 기록을 검사합니다

앞에서 선택한 대출과 주차 기록을 검사한 뒤 모델 입력에 사용합니다.

In [15]:
# 앞에서 만든 selected를 검사해 빈 근거 문제가 해결됐는지 확인합니다.
# 반복문으로 실제 선택 순서까지 검증합니다.
require_context(selected)
selected_order = []
for row in selected:
    selected_order.append(row["id"])
assert selected_order == ["E01-p3", "E01-p5"]
print("수정 후 선택한 기록:", len(selected))

수정 후 선택한 기록: 2
